# API Quickstart

A small in-memory example using `validate()` and `ReportView`.

This notebook stays intentionally close to the public API so it is easy to adapt into tests or application code.

In [1]:
from data_validator import ReportView, validate

In [2]:
data = [
    {"id": "1", "email": "alice@example.com", "age": "30", "status": "active"},
    {"id": "2", "email": "", "age": "17", "status": "active"},
    {"id": "2", "email": "not-an-email", "age": "unknown", "status": "inactive"},
]

checks = [
    {"columns_check": {"params": ["id", "email", "age", "status"]}},
    {"non_empty_check": {"params": ["id", "email"]}},
    {"unique_check": {"params": ["id"]}},
    {"types_check": {"params": {"age": "integer"}}},
    {"range_check": {"params": {"age": {"min": 18, "max": 99}}}},
    {"regex_check": {"params": {"email": r"^[^@\s]+@[^@\s]+\.[^@\s]+$"}}},
]

In [3]:
report = validate(data, checks=checks)
view = ReportView(report)

In [4]:
report.overall_passed, report.total_elapsed_ms

(False, 5.838)

In [5]:
view.summary()

check,passed,errors,elapsed_ms
str,bool,i64,f64
"""columns_check""",true,0,0.003
"""non_empty_check""",false,1,1.552
"""unique_check""",false,1,2.058
"""types_check""",false,1,0.556
"""range_check""",false,2,0.723
"""regex_check""",false,1,0.888


In [6]:
view.errors_df()

check,row,column,message,value
str,i64,str,str,str
"""non_empty_check""",2,"""email""","""Empty value in required column…",""""""
"""unique_check""",3,"""id""","""Duplicate value""","""2"""
"""types_check""",3,"""age""","""Expected type 'integer'""","""unknown"""
"""range_check""",3,"""age""","""Cannot parse as number for ran…","""unknown"""
"""range_check""",2,"""age""","""Below minimum 18.0""","""17"""
"""regex_check""",3,"""email""","""Value does not match pattern '…","""not-an-email"""


In [7]:
view

ReportView(FAIL, 6 checks)

In [8]:
report.model_dump()

{'file_path': '<in-memory>',
 'schema_path': '<inline>',
 'overall_passed': False,
 'results': [{'validator_name': 'columns_check',
   'passed': True,
   'errors': [],
   'elapsed_ms': 0.003},
  {'validator_name': 'non_empty_check',
   'passed': False,
   'errors': [{'row': 2,
     'column': 'email',
     'message': "Empty value in required column 'email'",
     'value': ''}],
   'elapsed_ms': 1.552},
  {'validator_name': 'unique_check',
   'passed': False,
   'errors': [{'row': 3,
     'column': 'id',
     'message': 'Duplicate value',
     'value': '2'}],
   'elapsed_ms': 2.058},
  {'validator_name': 'types_check',
   'passed': False,
   'errors': [{'row': 3,
     'column': 'age',
     'message': "Expected type 'integer'",
     'value': 'unknown'}],
   'elapsed_ms': 0.556},
  {'validator_name': 'range_check',
   'passed': False,
   'errors': [{'row': 3,
     'column': 'age',
     'message': 'Cannot parse as number for range check',
     'value': 'unknown'},
    {'row': 2,
     'colum